# Extraction et analyse de feedback avec l'OCR Mistral

Ce notebook remplace Tesseract (peu fiable sur les photos de texte manuscrit ou de mauvaise qualité) par **l'API OCR de Mistral** (`mistral-ocr-latest`), puis utilise un modèle de chat Mistral pour extraire, pour chaque feedback détecté :
- les **points positifs**
- les **points négatifs**
- les **attentes** exprimées

**Étapes :** installation → upload de l'image → OCR Mistral → analyse LLM → résultat structuré (JSON).

## 1. Installation des dépendances

In [17]:
!pip install -q pillow requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [18]:
import base64
import json
import re
import mimetypes
import requests
from getpass import getpass
from IPython.display import display, Markdown

## 3. Clé API Mistral

⚠️ **Ne codez jamais votre clé API en dur dans le notebook**, surtout si vous le partagez ou le publiez (c'était le cas dans la version précédente — pensez à révoquer cette ancienne clé depuis votre compte Mistral et à en générer une nouvelle).

In [19]:
MISTRAL_API_KEY = getpass("Entrez votre clé API Mistral : ")

## 4. Chemin de l'image locale

In [20]:
import os

# Vous pouvez spécifier le chemin direct de l'image ici :
image_path = "sample_feedback.jpg"

# Si le fichier n'existe pas, on demande le chemin à l'utilisateur
if not os.path.exists(image_path):
    image_path = input("Veuillez entrer le chemin de l'image locale (ex: feedback.jpg) : ").strip()

if os.path.exists(image_path):
    print("Image locale chargée :", image_path)
else:
    print("⚠️ Fichier non trouvé. Veuillez vérifier le chemin :", image_path)

Image locale chargée : sample_feedback.jpg


## 5. Extraction du texte avec l'OCR Mistral

On encode l'image en base64, puis on appelle l'endpoint `/v1/ocr` de Mistral avec le modèle `mistral-ocr-latest`. Ce modèle est bien plus performant que Tesseract sur les photos (angles, éclairage, écriture manuscrite, etc.).

In [21]:
def encode_image_to_data_uri(image_path):
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/jpeg"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"


def ocr_with_mistral(image_path, api_key=MISTRAL_API_KEY):
    url = "https://api.mistral.ai/v1/ocr"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "mistral-ocr-latest",
        "document": {
            "type": "image_url",
            "image_url": encode_image_to_data_uri(image_path)
        },
        "include_image_base64": False
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        raise RuntimeError(f"Erreur OCR Mistral ({response.status_code}): {response.text}")

    data = response.json()
    pages_text = [page.get("markdown", "") for page in data.get("pages", [])]
    return "\n\n".join(pages_text).strip()

In [22]:
# Lancer l'OCR
raw_text = ocr_with_mistral(image_path)
print("TEXTE EXTRAIT PAR L'OCR MISTRAL :\n")
print(raw_text)

TEXTE EXTRAIT PAR L'OCR MISTRAL :

- by the end of the program I'm hopeful and want to learn about all those complex concepts, enrich my knowledge and most important, be able to develop the capacity of being able to explain an idea clearly from start to finish. I definitely need to work on that. (Learn how to communicate my thoughts.)
- I expect to be able to understand most things about AI, blackchains etc. - be less timid and be a member that the group can rely on.


## 6. Analyse du feedback avec un modèle de chat Mistral

On envoie le texte extrait à `mistral-large-latest` avec des consignes strictes pour identifier, feedback par feedback : points positifs, points négatifs et attentes.

In [23]:

def analyze_feedback_with_mistral(text, api_key=MISTRAL_API_KEY):
    url = "https://api.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    prompt = f"""Tu es un système strict d'extraction d'information, spécialisé dans l'analyse de feedbacks.

TÂCHE :
À partir du texte ci-dessous (issu d'un OCR, peut contenir quelques imperfections), identifie un ou plusieurs feedbacks distincts.
Pour CHAQUE feedback, extrais :
- "points_positifs" : les éléments positifs exprimés
- "points_negatifs" : les éléments négatifs ou critiques exprimés
- "attentes" : les attentes, souhaits ou besoins exprimés par l'auteur (ce qu'il espère, souhaite apprendre, voir ou obtenir)

RÈGLES STRICTES :
- N'extrais que des phrases ou clauses complètes et cohérentes
- N'extrais jamais des mots isolés ou des fragments incompréhensibles
- Ignore le texte illisible ou les artefacts d'OCR
- Ne déduis rien qui ne soit pas explicitement présent dans le texte
- Si une catégorie n'est pas présente, laisse le tableau vide
- S'il n'y a qu'un seul feedback dans le texte, renvoie une liste avec un seul élément

FORMAT DE SORTIE : réponds UNIQUEMENT avec du JSON valide (pas de markdown, pas de balises ```)

{{
  "feedbacks": [
    {{
      "resume": "résumé en une phrase",
      "points_positifs": [],
      "points_negatifs": [],
      "attentes": []
    }}
  ]
}}

TEXTE :
{text}
"""

    payload = {
        "model": "mistral-large-latest",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        raise RuntimeError(f"Erreur chat Mistral ({response.status_code}): {response.text}")

    return response.json()["choices"][0]["message"]["content"]

## 7. Parsing JSON sécurisé

In [24]:
def parse_json_safe(text):
    text = re.sub(r"```json", "", text)
    text = re.sub(r"```", "", text)
    text = text.strip()
    return json.loads(text)

## 8. Exécution complète du pipeline

In [25]:
# 1. Analyse du texte OCR extrait précédemment
result = analyze_feedback_with_mistral(raw_text)
print("SORTIE BRUTE DU MODÈLE :\n")
print(result)

# 2. Parsing JSON
try:
    structured = parse_json_safe(result)
    print("\nJSON STRUCTURÉ :\n")
    print(json.dumps(structured, indent=2, ensure_ascii=False))
except Exception as e:
    print("Erreur de parsing JSON :", e)
    print(result)

SORTIE BRUTE DU MODÈLE :

{
  "feedbacks": [
    {
      "resume": "L'auteur exprime ses espoirs et attentes concernant son apprentissage et son développement personnel à la fin du programme.",
      "points_positifs": [
        "I'm hopeful and want to learn about all those complex concepts",
        "enrich my knowledge"
      ],
      "points_negatifs": [
        "I definitely need to work on that. (Learn how to communicate my thoughts.)"
      ],
      "attentes": [
        "be able to develop the capacity of being able to explain an idea clearly from start to finish",
        "understand most things about AI, blackchains etc",
        "be less timid and be a member that the group can rely on"
      ]
    }
  ]
}

JSON STRUCTURÉ :

{
  "feedbacks": [
    {
      "resume": "L'auteur exprime ses espoirs et attentes concernant son apprentissage et son développement personnel à la fin du programme.",
      "points_positifs": [
        "I'm hopeful and want to learn about all those comp

## 9. Affichage lisible du résultat

In [ ]:
from IPython.core.display import Markdown
from IPython import display
if 'structured' in globals():
    display(Markdown(f"## Texte extrait de la photo\n\n{raw_text}\n---"))
    for i, fb in enumerate(structured.get("feedbacks", []), start=1):
        md = f"### Feedback {i}\n"
        md += f"**Résumé :** {fb.get('resume', '')}\n\n"

        md += "**✅ Points positifs :**\n"
        pts = fb.get("points_positifs") or []
        md += "\n".join(f"- {p}" for p in pts) if pts else "- (aucun)"
        md += "\n\n"
        
        md += "**❌ Points négatifs :**\n"
        pts = fb.get("points_negatifs") or []
        md += "\n".join(f"- {p}" for p in pts) if pts else "- (aucun)"
        md += "\n\n"
        
        md += "**🎯 Attentes :**\n"
        pts = fb.get("attentes") or []
        md += "\n".join(f"- {p}" for p in pts) if pts else "- (aucune)"
        
        display(Markdown(md))

### Feedback 1
**Résumé :** L'auteur exprime ses espoirs et attentes concernant son apprentissage et son développement personnel à la fin du programme.

**✅ Points positifs :**
- I'm hopeful and want to learn about all those complex concepts
- enrich my knowledge

**❌ Points négatifs :**
- I definitely need to work on that. (Learn how to communicate my thoughts.)

**🎯 Attentes :**
- be able to develop the capacity of being able to explain an idea clearly from start to finish
- understand most things about AI, blackchains etc
- be less timid and be a member that the group can rely on